In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from pathlib import Path
from typing import Optional, Dict
import re
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# =====================================================
# PATHS (NMF VERSION)
# =====================================================

BASE_DIR = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/"
    "NMF-0126/result/NMF_Single_Injection"
)

ORIG_DIR = BASE_DIR
SEARCH_ROOT = BASE_DIR

OUT_BASE = Path(
    "/home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/"
    "NMF-0126/result/figures"
)

OUT_BASE.mkdir(parents=True, exist_ok=True)

K_LIST = [15, 25, 35]
N_LIST = [1215, 2000]

INJECTION_PATTERN = "f_*_{n}u_pos5_neg1_all_{k}recommendation.csv"

# =====================================================
# GENRE NORMALIZATION (ROBUST)
# =====================================================

def clean_basic(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("’", "'")
    s = re.sub(r"[_\s]+", " ", s)
    return s

GENRE_MAP = {
    "childrens": "children's",
    "children s": "children's",
    "children": "children's",
    "children's": "children's",
    "children_s": "children's",
    "childrens books": "children's",
    "kids": "children's",
    "kids books": "children's",
}

def norm_genre(s: str) -> str:
    s = clean_basic(s)
    s_no_apos = s.replace("'", "")
    return GENRE_MAP.get(s_no_apos, s)

def slugify(x: str) -> str:
    return re.sub(r"_+","_", re.sub(r"[^A-Za-z0-9]+","_",str(x))).strip("_").lower()

# =====================================================
# IO HELPERS
# =====================================================

def load_rec_csv(fp):
    df = pd.read_csv(fp, low_memory=False)
    if "genres_all" not in df.columns:
        df["genres_all"] = ""
    return df

def has_genre(cell, target):
    if pd.isna(cell):
        return False
    for t in str(cell).split(","):
        if norm_genre(t) == target:
            return True
    return False

def extract_genre_from_filename(p: Path) -> Optional[str]:

    m = re.match(
        r"^f_(.+?)_(\d+)u_pos\d+_neg[^_]+_all_(\d+)recommendation\.csv$",
        p.name,
        flags=re.IGNORECASE
    )

    if not m:
        return None

    return norm_genre(m.group(1))

def find_injection(genre, n, k):

    patt = INJECTION_PATTERN.format(n=n, k=k)

    for p in SEARCH_ROOT.glob(patt):
        g = extract_genre_from_filename(p)
        if g == genre:
            return p

    return None

# =====================================================
# DETECT ALL GENRES
# =====================================================

genres = set()

for p in SEARCH_ROOT.glob("f_*recommendation.csv"):
    g = extract_genre_from_filename(p)
    if g:
        genres.add(g)

genres = sorted(genres)

print("Detected genres:")
for g in genres:
    print(" -", g)

# =====================================================
# PLOTTING
# =====================================================

def plot_genre(genre, data_by_k, out_png):

    ks = sorted(data_by_k.keys())

    groups = ["Original"] + [str(n) for n in N_LIST]

    width = 0.8 / len(groups)
    x = list(range(len(ks)))

    plt.figure(figsize=(8.8,4.4), dpi=160)

    for j,g in enumerate(groups):

        offs = [i + (j-(len(groups)-1)/2)*width for i in x]
        vals = [data_by_k[k].get(g,0) for k in ks]

        plt.bar(
            offs,
            vals,
            width=width,
            label=("n="+g if g!="Original" else "Original")
        )

    plt.xticks(x, [f"Top-{k}" for k in ks])
    plt.ylabel("Avg # of genre books per user")
    plt.title(f"NMF – {genre.capitalize()}")
    plt.legend(fontsize=9, ncol=4)
    plt.tight_layout()

    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png)
    plt.close()

# =====================================================
# MAIN
# =====================================================

def main():

    for genre in genres:

        print("\nProcessing:", genre)

        out_dir = OUT_BASE / f"{slugify(genre)}_explanation"
        out_dir.mkdir(parents=True, exist_ok=True)

        data_by_k: Dict[int, Dict[str,float]] = {k:{} for k in K_LIST}

        txt_out = out_dir / "explanation.txt"

        with open(txt_out, "w") as f:

            f.write(f"{genre}:\n")

            # ---------- ORIGINAL ----------

            for K in K_LIST:

                orig_fp = ORIG_DIR / f"ORIGINAL_{K}recommendation.csv"

                if not orig_fp.exists():
                    continue

                df = load_rec_csv(orig_fp)

                total_users = df["user_id"].nunique()

                mask = df["genres_all"].apply(lambda x: has_genre(x, genre))

                freq = mask.sum()

                avg = freq / total_users if total_users > 0 else 0

                data_by_k[K]["Original"] = avg

                f.write(f"original {K}: avg_per_user: {avg:.6f}\n")

            # ---------- NEW INJECTIONS ----------

            for K in K_LIST:
                for N in N_LIST:

                    fp = find_injection(genre, N, K)

                    if not fp:
                        continue

                    df = load_rec_csv(fp)

                    total_users = df["user_id"].nunique()

                    mask = df["genres_all"].apply(lambda x: has_genre(x, genre))

                    freq = mask.sum()

                    avg = freq / total_users if total_users > 0 else 0

                    data_by_k[K][str(N)] = avg

                    f.write(f"{N}u, {K}, avg_per_user: {avg:.6f}\n")

        # ---------- FIG ----------

        fig_out = out_dir / f"{slugify(genre)}__nmf.png"

        plot_genre(genre, data_by_k, fig_out)

        print("Saved:", out_dir)

    print("\n✅ ALL GENRES DONE (NMF)")

# =====================================================

if __name__ == "__main__":
    main()


Detected genres:
 - adult
 - adventure
 - children's
 - classics
 - drama
 - fantasy
 - historical
 - horror
 - mystery
 - nonfiction
 - romance
 - science fiction
 - thriller

Processing: adult
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/NMF-0126/result/figures/adult_explanation

Processing: adventure
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/NMF-0126/result/figures/adventure_explanation

Processing: children's
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/NMF-0126/result/figures/children_s_explanation

Processing: classics
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/NMF-0126/result/figures/classics_explanation

Processing: drama
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/NMF-0126/result/figures/drama_explanation

Processing: fantasy
Saved: /home/moshtasa/Research/phd-svd-recsys/Book/0126_similar_pr/NMF-0126/result/figures/fantasy_explanation

Processing: historical
Save